<a href="https://colab.research.google.com/github/AICHUCKY/Ai-with-Chucky-Colab-Notebooks/blob/main/Idol%203d%20Colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ⚡ Idol 3D • Image to 3D Gaussian Splatting

### 🔴 **Brought to you by [AI With Chucky](https://youtube.com/@AIWithChucky)**

Transform single 2D images into interactive 3D Gaussian Splat models with real-time 3D viewport rendering.

- **Engine**: True 3D Gaussian Splatting renderer (per-splat covariance, opacity and depth-sorted blending) with interactive orbit/pan/zoom.
- **Ingestion**: Astral `uv` package resolver + 16-connection parallel `aria2c` model downloader.
- **Turing Optimizations**: FP16 DINOv3 + memory-efficient SDPA attention for Tesla T4.
- **Non-Blocking Tunnel**: Cloudflare Tunnel with asynchronous polling (immune to 524 timeouts).

In [ ]:
# @title 🚀 1. Accelerated Environment Setup & Model Ingestion

import os
import sys
import time
import json
import shutil
import subprocess

BUNDLE_REPO = "AiWithChucky/idol-models-bundle"
LOCAL_BUNDLE_DIR = "/content/idol_models_bundle"

def print_header(title):
    w = 66
    print(f"\n\033[1;36m┌{'─' * w}┐\033[0m")
    print(f"\033[1;36m│\033[0m  \033[1;37m{title:<{w - 2}}\033[0m\033[1;36m│\033[0m")
    print(f"\033[1;36m└{'─' * w}┘\033[0m\n")

def log_msg(tag, msg, color="32"):
    print(f"\033[1;{color}m[{tag.center(10)}]\033[0m {msg}")

def run(cmd, **kw):
    """subprocess.run with a readable error instead of a bare CalledProcessError."""
    try:
        return subprocess.run(cmd, check=True, **kw)
    except subprocess.CalledProcessError as e:
        log_msg("ERROR", f"Command failed ({e.returncode}): {' '.join(map(str, cmd))}", "31")
        raise

t0 = time.time()
print_header("IDOL 3D • ENVIRONMENT INGESTION")

# torch must NOT be imported before the SDK install: the install can change the torch
# build on disk, and an already-imported torch would then be stale/mismatched.
if "torch" in sys.modules:
    log_msg("WARN", "torch was already imported in this session. If anything breaks later, "
                    "use Runtime > Restart session and run this cell first.", "33")

# 0. GPU check without importing torch
try:
    gpu = subprocess.run(
        ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
        capture_output=True, text=True, check=True
    ).stdout.strip().splitlines()[0]
    log_msg("HARDWARE", gpu, "35")
except (FileNotFoundError, subprocess.CalledProcessError, IndexError):
    log_msg("ERROR", "No CUDA device found. Switch Colab runtime to T4 GPU "
                     "(Runtime > Change runtime type).", "31")
    raise SystemError("CUDA GPU required")

# 1. Install uv, aria2c, and cloudflared
log_msg("TOOLS", "Configuring aria2c, Astral uv, and Cloudflare Tunnel...")
if not shutil.which("aria2c"):
    subprocess.run(["apt-get", "-qq", "update"], check=False)
    run(["apt-get", "-qq", "install", "-y", "aria2"])
run([sys.executable, "-m", "pip", "install", "-q", "uv"])

if not os.path.exists("/usr/local/bin/cloudflared"):
    # -f: fail on HTTP errors instead of saving an HTML error page as the "binary"
    run([
        "curl", "-fsSL", "--retry", "3", "-o", "/usr/local/bin/cloudflared",
        "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"
    ])
    run(["chmod", "+x", "/usr/local/bin/cloudflared"])

# 2. Clone Idol SDK
if not os.path.exists("/content/idol-sdk"):
    log_msg("GIT", "Cloning speridlabs/idol-sdk (depth=1)...", "34")
    run(["git", "clone", "--depth=1", "https://github.com/speridlabs/idol-sdk.git", "/content/idol-sdk"])
os.chdir("/content/idol-sdk")

# 3. Fast pip install via uv
log_msg("UV", "Resolving dependencies via uv...")
run([
    sys.executable, "-m", "uv", "pip", "install", "--system", "-q",
    "-e", ".", "huggingface_hub", "flask", "pillow", "numpy"
])

# 4. High-speed multi-connection ingestion with aria2c
def cache_is_complete():
    if not os.path.exists(os.path.join(LOCAL_BUNDLE_DIR, "idol", "denoiser.safetensors")):
        return False
    # aria2 leaves *.aria2 control files next to unfinished downloads
    for root, _, fs in os.walk(LOCAL_BUNDLE_DIR):
        if any(f.endswith(".aria2") for f in fs):
            return False
    return True

os.makedirs(LOCAL_BUNDLE_DIR, exist_ok=True)
if not cache_is_complete():
    log_msg("ARIA2C", f"Starting 16-connection parallel ingestion from {BUNDLE_REPO}...")
    from huggingface_hub import HfApi
    files = HfApi().list_repo_files(repo_id=BUNDLE_REPO)

    aria_file = "/content/aria_tasks.txt"
    with open(aria_file, "w") as f:
        for fp in files:
            if os.path.basename(fp).startswith(".") or fp.endswith(".gitattributes"):
                continue
            url = f"https://huggingface.co/{BUNDLE_REPO}/resolve/main/{fp}"
            d = os.path.join(LOCAL_BUNDLE_DIR, os.path.dirname(fp))
            f.write(f"{url}\n  dir={d}\n  out={os.path.basename(fp)}\n")

    run([
        "aria2c", "-c", "-x", "16", "-s", "16", "-j", "8", "-k", "1M",
        "--max-tries=5", "--retry-wait=3", "--auto-file-renaming=false",
        "--file-allocation=none", "--summary-interval=5", f"--input-file={aria_file}"
    ])
    if not cache_is_complete():
        raise RuntimeError("Model download did not complete (missing denoiser or leftover .aria2 files). Re-run this cell.")
    log_msg("ARIA2C", "Checkpoints successfully downloaded.", "32")
else:
    log_msg("CACHE", "Checkpoints found in local cache.", "32")

# 5. Configure local config.json
config_path = os.path.join(LOCAL_BUNDLE_DIR, "idol", "config.json")
if os.path.exists(config_path):
    with open(config_path, "r") as f:
        cfg = json.load(f)
    enc = cfg.setdefault("image_encoders", {})
    enc["dinov3"] = os.path.join(LOCAL_BUNDLE_DIR, "dinov3")
    enc["flux2_vae"] = os.path.join(LOCAL_BUNDLE_DIR, "flux2_vae")
    with open(config_path, "w") as f:
        json.dump(cfg, f, indent=2)
    log_msg("CONFIG", "Mapped encoder weights to local directory.", "32")
else:
    log_msg("WARN", f"{config_path} not found - the model bundle layout may have changed.", "33")

# 6. Final sanity check (torch is imported only now, after all installs)
import torch
if not torch.cuda.is_available():
    raise SystemError("PyTorch cannot see the GPU. Restart the session (Runtime > Restart session) and re-run.")
vram = round(torch.cuda.get_device_properties(0).total_memory / (1024**3), 2)
log_msg("TORCH", f"torch {torch.__version__} | {torch.cuda.get_device_name(0)} ({vram} GB VRAM)", "35")

print(f"\n\033[1;32m✓ Setup completed in {round(time.time() - t0, 1)}s\033[0m\n")


### Step 2: Start Idol 3D WebUI & 3D Viewer

In [ ]:
# @title 🎨 2. Launch Idol 3D WebUI

PORT = 5000
MAX_KEPT_MODELS = 6          # old .ply files beyond this are deleted to protect disk space

import os
import gc
import re
import sys
import time
import uuid
import glob
import logging
import inspect
import threading
import traceback
import subprocess
import torch
from flask import Flask, request, jsonify, Response, send_from_directory
from werkzeug.serving import make_server
from PIL import Image, ImageOps

# 1. Quiet the dev-server request log
logging.getLogger('werkzeug').setLevel(logging.ERROR)

# 2. Stop a previous run of this cell (frees the port + kills the old tunnel)
for _name, _stop in (("_idol_server", "shutdown"), ("_idol_tunnel", "terminate")):
    _old = globals().get(_name)
    if _old is not None:
        try:
            getattr(_old, _stop)()
        except Exception:
            pass

# 3. Attention configuration for Turing (T4)
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)
torch.backends.cuda.matmul.allow_tf32 = True

# 4. Bypass remote check
import idol.pipeline
import idol.conditioning
idol.pipeline.require_encoder_access = lambda repos: None
idol.conditioning.require_encoder_access = lambda repos: None

from idol import IdolPipeline

UPLOAD_DIR = "/content/webui_uploads"
OUTPUT_DIR = "/content/webui_outputs"
os.makedirs(UPLOAD_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Re-running this cell reuses the already-loaded pipeline instead of reloading it into VRAM
if globals().get("pipeline") is None:
    print("\033[1;34m[INIT]\033[0m Loading IdolPipeline into GPU memory...")
    pipeline = IdolPipeline.from_pretrained(
        "/content/idol_models_bundle/idol",
        optimize=False,
        dinov3_dtype="fp16"
    )
    print("\033[1;32m[INIT]\033[0m Pipeline active and cached in VRAM!")
else:
    print("\033[1;32m[INIT]\033[0m Reusing pipeline already loaded in VRAM.")

# 5. Async GPU job registry (one GPU job at a time; extra jobs wait in 'queued')
TASKS = {}
TASK_LOCK = threading.Lock()
GPU_LOCK = threading.Lock()

def set_task(task_id, **data):
    with TASK_LOCK:
        TASKS[task_id] = data
        if len(TASKS) > 200:                       # keep the registry bounded
            for k in list(TASKS)[:len(TASKS) - 200]:
                TASKS.pop(k, None)

def prune_outputs():
    files = sorted(glob.glob(os.path.join(OUTPUT_DIR, "*.ply")), key=os.path.getmtime, reverse=True)
    for old in files[MAX_KEPT_MODELS:]:
        try:
            os.remove(old)
        except OSError:
            pass

def build_call_kwargs(input_path, mode, num_gaussians, steps, guidance_scale, time_shift, seed):
    """Map UI values onto whatever argument names this SDK version uses."""
    sig = inspect.signature(pipeline.__call__)
    params = sig.parameters
    var_kw = any(p.kind is inspect.Parameter.VAR_KEYWORD for p in params.values())

    def put(kw, candidates, value, required=False):
        for name in candidates:
            if name in params:
                kw[name] = value
                return
        if required and var_kw:
            kw[candidates[0]] = value           # **kwargs signature: trust the primary name
        elif required:
            raise TypeError(f"IdolPipeline.__call__ has no argument named any of {candidates}")
        else:
            print(f"\033[1;33m[WARN]\033[0m Pipeline has no '{candidates[0]}' argument - UI value ignored.")

    kw = {}
    put(kw, ["image"], input_path, required=True)
    put(kw, ["mode"], mode)
    put(kw, ["seed"], seed)
    put(kw, ["num_gaussians"], num_gaussians)
    put(kw, ["steps", "num_steps"], steps)
    put(kw, ["guidance_scale", "cfg_scale"], guidance_scale)
    put(kw, ["time_shift", "rescale_t"], time_shift)
    return kw

def run_gpu_generation(task_id, input_path, mode, num_gaussians, steps, guidance_scale, time_shift, seed):
    t_wait = time.time()
    ply_filename = f"model_{int(t_wait * 1000)}_{task_id}.ply"
    ply_path = os.path.join(OUTPUT_DIR, ply_filename)
    output = None

    with GPU_LOCK:
        set_task(task_id, status="processing")
        t0 = time.time()
        try:
            call_kwargs = build_call_kwargs(input_path, mode, num_gaussians, steps, guidance_scale, time_shift, seed)
            gc.collect()
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

            with torch.inference_mode():
                output = pipeline(**call_kwargs)
                output.gaussian.save_ply(ply_path)

            elapsed = round(time.time() - t0, 1)
            peak_vram = round(torch.cuda.max_memory_allocated() / (1024**3), 2)
            print(f"\033[1;32m[DONE]\033[0m Task {task_id} generated in {elapsed}s | VRAM: {peak_vram} GB")
            set_task(task_id, status="completed", ply_url=f"/files/{ply_filename}",
                     elapsed=elapsed, vram=peak_vram, gaussians=num_gaussians)
            prune_outputs()
        except torch.cuda.OutOfMemoryError:
            print(f"\033[1;31m[ERROR]\033[0m Task {task_id} ran out of GPU memory")
            set_task(task_id, status="failed",
                     error="GPU out of memory. Lower the Gaussian count or sampling steps and try again.")
        except Exception as e:
            traceback.print_exc()
            set_task(task_id, status="failed", error=f"{type(e).__name__}: {e}")
        finally:
            output = None
            gc.collect()
            torch.cuda.empty_cache()
            try:
                os.remove(input_path)
            except OSError:
                pass

app = Flask(__name__)
app.config["MAX_CONTENT_LENGTH"] = 64 * 1024 * 1024
app.config["SEND_FILE_MAX_AGE_DEFAULT"] = 3600

HTML_STUDIO = r"""<!DOCTYPE html>
<html lang="en">
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>Idol 3D</title>
    <style>
        :root {
            --bg: #05070c;
            --card: #0b0f19;
            --border: #151d2f;
            --primary: #38bdf8;
            --primary-glow: rgba(56, 189, 248, 0.25);
            --accent: #10b981;
            --text: #f8fafc;
            --dim: #94a3b8;
        }
        * { box-sizing: border-box; margin: 0; padding: 0; font-family: -apple-system, BlinkMacSystemFont, 'Inter', 'Segoe UI', Roboto, sans-serif; }
        body { background: var(--bg); color: var(--text); min-height: 100vh; padding: 18px; display: flex; justify-content: center; overflow-x: hidden; }
        .layout { max-width: 1440px; width: 100%; display: grid; grid-template-columns: 430px 1fr; gap: 20px; align-items: start; }
        @media (max-width: 1024px) { .layout { grid-template-columns: 1fr; } }

        .card { background: var(--card); border: 1px solid var(--border); border-radius: 14px; padding: 22px; display: flex; flex-direction: column; box-shadow: 0 10px 30px rgba(0,0,0,0.5); min-width: 0; }
        .title-row { display: flex; align-items: center; justify-content: space-between; margin-bottom: 2px; }
        h1 { font-size: 20px; font-weight: 700; color: #fff; }
        h1 span { color: var(--primary); }
        .badge { background: rgba(56,189,248,0.12); color: var(--primary); font-size: 11px; padding: 3px 8px; border-radius: 20px; font-weight: 600; }
        p.desc { font-size: 13px; color: var(--dim); margin-bottom: 6px; }
        .creator-credit { font-size: 12px; font-weight: 600; color: #cbd5e1; margin-bottom: 14px; display: inline-flex; align-items: center; gap: 5px; }
        .creator-credit a { color: var(--primary); text-decoration: underline; text-underline-offset: 2px; }

        .dropzone { border: 2px dashed var(--border); border-radius: 10px; padding: 18px 12px; text-align: center; cursor: pointer; transition: 0.15s; background: rgba(0,0,0,0.3); margin-bottom: 14px; }
        .dropzone:hover, .dropzone.drag { border-color: var(--primary); background: var(--primary-glow); }
        #preview { max-height: 180px; max-width: 100%; border-radius: 8px; margin-top: 10px; display: none; margin-left: auto; margin-right: auto; object-fit: contain; }

        .presets { display: flex; gap: 8px; margin-bottom: 14px; }
        .preset-btn { flex: 1; background: #131a2c; border: 1px solid var(--border); color: #cbd5e1; border-radius: 6px; padding: 7px 4px; font-size: 11px; font-weight: 600; cursor: pointer; transition: 0.15s; }
        .preset-btn:hover { border-color: var(--primary); color: #fff; background: #1a2338; }

        .grid-2 { display: grid; grid-template-columns: 1fr 1fr; gap: 12px; margin-bottom: 12px; }
        label { display: flex; justify-content: space-between; font-size: 12px; font-weight: 600; color: #cbd5e1; margin-bottom: 5px; }
        label span.val { color: var(--primary); font-family: monospace; }
        select, input[type="number"], input[type="range"] { width: 100%; background: #07090e; border: 1px solid var(--border); color: #fff; border-radius: 6px; padding: 8px 10px; font-size: 13px; }
        input[type="range"] { padding: 4px 0; accent-color: var(--primary); cursor: pointer; }

        button.btn-gen { background: #0284c7; color: #fff; border: none; padding: 13px; border-radius: 8px; font-size: 15px; font-weight: 600; cursor: pointer; transition: 0.2s; margin-top: 8px; width: 100%; box-shadow: 0 4px 15px var(--primary-glow); }
        button.btn-gen:hover { background: #0369a1; transform: translateY(-1px); }
        button.btn-gen:disabled { background: #1e293b; color: #64748b; cursor: not-allowed; transform: none; box-shadow: none; }
        .status { margin-top: 10px; font-size: 12px; color: var(--primary); min-height: 18px; text-align: center; font-weight: 500; word-break: break-word; }

        /* 3D Viewport Area */
        .viewport-container { width: 100%; height: 640px; max-height: 75vh; min-height: 320px; background: #04060a; border-radius: 12px; border: 1px solid var(--border); position: relative; overflow: hidden; display: flex; align-items: center; justify-content: center; }
        #playerFrame { width: 100%; height: 100%; border: none; display: none; background: #04060a; }
        .viewport-empty { text-align: center; color: #475569; position: absolute; pointer-events: none; padding: 0 20px; }
        .viewport-empty svg { width: 52px; height: 52px; margin-bottom: 10px; opacity: 0.35; stroke: currentColor; }

        .hud-stats { position: absolute; top: 12px; left: 12px; background: rgba(11, 15, 25, 0.85); border: 1px solid rgba(255,255,255,0.1); border-radius: 6px; padding: 6px 12px; font-size: 11px; color: var(--dim); backdrop-filter: blur(4px); pointer-events: none; display: none; line-height: 1.4; z-index: 5; }
        .hud-stats span { color: var(--primary); font-weight: 600; font-family: monospace; }

        .actions-bar { display: flex; gap: 10px; margin-top: 14px; flex-wrap: wrap; }
        .action-link { flex: 1; min-width: 180px; text-align: center; background: #059669; border: 1px solid #059669; color: #fff; padding: 12px; border-radius: 8px; text-decoration: none; font-size: 14px; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 6px; transition: 0.2s; }
        .action-link:hover { background: #047857; }
        .action-link.secondary { background: #131a2c; border-color: var(--border); }
        .action-link.secondary:hover { border-color: var(--primary); color: var(--primary); }
    </style>
</head>
<body>
    <div class="layout">
        <!-- Left: Workbench -->
        <div class="card">
            <div class="title-row">
                <h1><span>Idol</span> 3D</h1>
                <span class="badge">3DGS Engine</span>
            </div>
            <p class="desc">Single 2D Image to Radiance Gaussian Splatting</p>
            <div class="creator-credit">🔴 Brought to you by <a href="https://youtube.com/@AIWithChucky" target="_blank" rel="noopener">AI With Chucky</a></div>

            <div class="dropzone" id="dropzone">
                <p id="dropMsg" style="font-size: 13px; color: var(--dim);">Drag &amp; drop image or click to browse</p>
                <img id="preview" alt="Uploaded Image">
            </div>
            <input type="file" id="fileIn" accept="image/*" style="display:none">

            <div class="presets">
                <button class="preset-btn" onclick="applyPreset(20, 131072, 2.5)">⚡ Turbo</button>
                <button class="preset-btn" onclick="applyPreset(35, 262144, 3.0)">⚖️ Balanced</button>
                <button class="preset-btn" onclick="applyPreset(50, 524288, 3.5)">💎 Quality</button>
            </div>

            <div class="grid-2">
                <div>
                    <label for="mode">Framing Mode</label>
                    <select id="mode">
                        <option value="real" selected>Real Photograph</option>
                        <option value="synthetic">3D Render / AI Art</option>
                    </select>
                </div>
                <div>
                    <label for="gaussians">Gaussians</label>
                    <select id="gaussians">
                        <option value="65536">65,536</option>
                        <option value="131072">131,072</option>
                        <option value="262144" selected>262,144</option>
                        <option value="524288">524,288</option>
                        <option value="1048576">1,048,576</option>
                    </select>
                </div>
            </div>

            <div style="margin-bottom: 12px;">
                <label for="steps">Euler Sampling Steps <span class="val" id="stepsVal">35</span></label>
                <input type="range" id="steps" min="15" max="75" value="35" step="1" oninput="document.getElementById('stepsVal').innerText=this.value">
            </div>

            <div class="grid-2">
                <div>
                    <label for="cfg">CFG Scale <span class="val" id="cfgVal">3.0</span></label>
                    <input type="range" id="cfg" min="1.0" max="7.0" value="3.0" step="0.1" oninput="document.getElementById('cfgVal').innerText=parseFloat(this.value).toFixed(1)">
                </div>
                <div>
                    <label for="timeShift">Time Shift (&tau;) <span class="val" id="shiftVal">3.0</span></label>
                    <input type="range" id="timeShift" min="1.0" max="5.0" value="3.0" step="0.1" oninput="document.getElementById('shiftVal').innerText=parseFloat(this.value).toFixed(1)">
                </div>
            </div>

            <div style="margin-bottom: 6px;">
                <label for="seed">Seed (-1 for random)</label>
                <input type="number" id="seed" value="42" step="1">
            </div>

            <button class="btn-gen" id="genBtn" onclick="generate()" disabled>Reconstruct 3D Model</button>
            <div class="status" id="statusMsg"></div>
        </div>

        <!-- Right: 3D Viewport Stage -->
        <div class="card">
            <div class="viewport-container" id="viewport">
                <div class="hud-stats" id="hudStats">
                    <div>Gaussians: <span id="hudCount">0</span></div>
                    <div>Generation Time: <span id="hudTime">0s</span></div>
                </div>

                <iframe id="playerFrame" title="3D Gaussian Splat Viewer"></iframe>

                <div class="viewport-empty" id="emptyNotice">
                    <svg viewBox="0 0 24 24" fill="none" stroke-width="1.5">
                        <path d="M21 16V8a2 2 0 0 0-1-1.73l-7-4a2 2 0 0 0-2 0l-7 4A2 2 0 0 0 3 8v8a2 2 0 0 0 1 1.73l7 4a2 2 0 0 0 2 0l7-4A2 2 0 0 0 21 16z"></path>
                        <polyline points="3.27 6.96 12 12.01 20.73 6.96"></polyline>
                        <line x1="12" y1="22.08" x2="12" y2="12"></line>
                    </svg>
                    <p>Load an image &amp; click Reconstruct to stream true 3D Gaussian Splats</p>
                </div>
            </div>

            <div class="actions-bar">
                <a id="dlPly" class="action-link dl" style="display:none;" href="#" download>Download Full .PLY Model</a>
                <a id="openEditor" class="action-link secondary" style="display:none;" href="#" target="_blank" rel="noopener">Open in Dedicated Viewer</a>
            </div>
        </div>
    </div>

    <script>
        let chosenFile = null;
        let pollToken = 0;

        window.applyPreset = function (steps, gaussians, cfg) {
            document.getElementById('steps').value = steps;
            document.getElementById('stepsVal').innerText = steps;
            document.getElementById('gaussians').value = gaussians;
            document.getElementById('cfg').value = cfg;
            document.getElementById('cfgVal').innerText = cfg.toFixed(1);
        };

        window.handleFile = function (f) {
            if (!f) return;
            if (!f.type || !f.type.startsWith('image/')) {
                document.getElementById('statusMsg').innerText = '❌ Please choose an image file.';
                return;
            }
            chosenFile = f;
            const r = new FileReader();
            r.onload = (e) => {
                const p = document.getElementById('preview');
                p.src = e.target.result;
                p.style.display = 'block';
                document.getElementById('dropMsg').innerText = f.name;
                document.getElementById('genBtn').disabled = false;
                document.getElementById('statusMsg').innerText = '';
            };
            r.readAsDataURL(f);
        };

        // Dropzone: click-to-browse + real drag & drop
        (function () {
            const dz = document.getElementById('dropzone');
            const fi = document.getElementById('fileIn');
            dz.addEventListener('click', () => fi.click());
            fi.addEventListener('change', () => { handleFile(fi.files[0]); fi.value = ''; });
            ['dragenter', 'dragover'].forEach(ev => dz.addEventListener(ev, e => { e.preventDefault(); dz.classList.add('drag'); }));
            ['dragleave', 'drop'].forEach(ev => dz.addEventListener(ev, e => { e.preventDefault(); dz.classList.remove('drag'); }));
            dz.addEventListener('drop', e => { if (e.dataTransfer && e.dataTransfer.files.length) handleFile(e.dataTransfer.files[0]); });
            // stop the browser from navigating to a file dropped outside the dropzone
            ['dragover', 'drop'].forEach(ev => window.addEventListener(ev, e => e.preventDefault()));
        })();

        function showResult(data) {
            const dl = document.getElementById('dlPly');
            dl.href = data.ply_url;
            dl.style.display = 'inline-flex';
            const ed = document.getElementById('openEditor');
            ed.href = '/viewer?model=' + encodeURIComponent(data.ply_url);
            ed.style.display = 'inline-flex';

            document.getElementById('hudCount').innerText = Number(data.gaussians).toLocaleString();
            document.getElementById('hudTime').innerText = data.elapsed + 's';
            document.getElementById('hudStats').style.display = 'block';
            document.getElementById('emptyNotice').style.display = 'none';
            const frame = document.getElementById('playerFrame');
            frame.src = '/viewer?model=' + encodeURIComponent(data.ply_url);
            frame.style.display = 'block';
        }

        window.generate = async function () {
            if (!chosenFile) return;
            const btn = document.getElementById('genBtn');
            const status = document.getElementById('statusMsg');
            const token = ++pollToken;

            btn.disabled = true;
            status.innerText = '⚡ Uploading image...';

            const fd = new FormData();
            fd.append('image', chosenFile);
            fd.append('mode', document.getElementById('mode').value);
            fd.append('num_gaussians', document.getElementById('gaussians').value);
            fd.append('steps', document.getElementById('steps').value);
            fd.append('guidance_scale', document.getElementById('cfg').value);
            fd.append('time_shift', document.getElementById('timeShift').value);
            fd.append('seed', document.getElementById('seed').value || '42');

            try {
                const submitRes = await fetch('/generate', { method: 'POST', body: fd });
                let submitData;
                try { submitData = await submitRes.json(); }
                catch (_) { throw new Error('Server returned HTTP ' + submitRes.status); }

                if (!submitData.success) {
                    status.innerText = '❌ Error: ' + submitData.error;
                    btn.disabled = false;
                    return;
                }

                const taskId = submitData.task_id;
                const startTime = Date.now();
                let failures = 0;

                const poll = async () => {
                    if (token !== pollToken) return;
                    const sec = Math.round((Date.now() - startTime) / 1000);
                    try {
                        const pollRes = await fetch('/status/' + taskId, { cache: 'no-store' });
                        const data = await pollRes.json();
                        failures = 0;
                        if (token !== pollToken) return;

                        if (data.status === 'queued') {
                            status.innerText = `⏳ Waiting for the GPU... (${sec}s)`;
                        } else if (data.status === 'processing') {
                            status.innerText = `⚡ Denoising 3D Gaussians on GPU... (${sec}s)`;
                        } else if (data.status === 'completed') {
                            btn.disabled = false;
                            status.innerText = `✓ Generated in ${data.elapsed}s (${data.vram} GB peak VRAM).`;
                            showResult(data);
                            return;
                        } else if (data.status === 'failed') {
                            btn.disabled = false;
                            status.innerText = '❌ Error: ' + data.error;
                            return;
                        }
                    } catch (e) {
                        failures++;
                        if (failures > 40) {
                            btn.disabled = false;
                            status.innerText = '❌ Lost connection to the server.';
                            return;
                        }
                        status.innerText = `⚡ Working... reconnecting (${sec}s)`;
                    }
                    setTimeout(poll, 1500);
                };
                poll();
            } catch (err) {
                status.innerText = '❌ Failed: ' + err.message;
                btn.disabled = false;
            }
        };
    </script>
</body>
</html>
"""

# Standalone native 3DGS viewer (true Gaussian splatting: per-splat covariance, opacity, depth-sorted blending)
HTML_VIEWER = r"""<!DOCTYPE html>
<html>
<head>
    <meta charset="UTF-8">
    <meta name="viewport" content="width=device-width, initial-scale=1.0">
    <title>3DGS Player</title>
    <script src="https://cdn.jsdelivr.net/npm/three@0.128.0/build/three.min.js"></script>
    <script src="https://cdn.jsdelivr.net/npm/three@0.128.0/examples/js/controls/OrbitControls.js"></script>
    <style>
        * { margin: 0; padding: 0; box-sizing: border-box; }
        html, body { width: 100%; height: 100%; background: #04060a; overflow: hidden; font-family: -apple-system, BlinkMacSystemFont, 'Inter', 'Segoe UI', sans-serif; }
        #canvas { position: absolute; inset: 0; width: 100%; height: 100%; display: block; touch-action: none; outline: none; }
        #loading { position: absolute; top: 50%; left: 50%; transform: translate(-50%, -50%); color: #38bdf8; font-size: 13px; font-weight: 600; letter-spacing: 0.4px; text-align: center; max-width: 80%; line-height: 1.5; z-index: 4; }
        #loading.error { color: #f87171; }
        .hint { position: absolute; bottom: 10px; left: 12px; color: rgba(255,255,255,0.4); font-size: 11px; pointer-events: none; z-index: 3; }
        #bar { position: absolute; top: 10px; right: 10px; display: none; gap: 6px; align-items: center; z-index: 5; background: rgba(11,15,25,0.85); border: 1px solid rgba(255,255,255,0.1); border-radius: 8px; padding: 5px 8px; backdrop-filter: blur(4px); color: #94a3b8; font-size: 11px; }
        #bar button { background: #131a2c; border: 1px solid #1f2a44; color: #cbd5e1; border-radius: 5px; padding: 4px 8px; font-size: 11px; font-weight: 600; cursor: pointer; }
        #bar button:hover { border-color: #38bdf8; color: #fff; }
        #bar button.on { background: #0284c7; border-color: #0284c7; color: #fff; }
        #bar input[type=range] { width: 70px; accent-color: #38bdf8; }
        #count { position: absolute; bottom: 10px; right: 12px; color: rgba(255,255,255,0.45); font-size: 11px; font-family: monospace; pointer-events: none; z-index: 3; }
    </style>
</head>
<body>
    <div id="loading">Loading 3D Gaussian Radiance Field...</div>
    <canvas id="canvas" tabindex="0"></canvas>
    <div id="bar">
        <button id="btnReset" title="Reset camera">Reset</button>
        <button id="btnRotate" title="Auto-rotate">Rotate</button>
        <button id="btnFlip" title="Flip model upside-down (fixes coordinate convention)">Flip</button>
        <span>Size</span><input type="range" id="sizeSlider" min="0.3" max="2.5" step="0.05" value="1">
    </div>
    <div class="hint">Rotate: Left Click &bull; Pan: Right Click / Shift+Drag &bull; Zoom: Scroll / Pinch</div>
    <div id="count"></div>

    <script>
    (function () {
        'use strict';

        const loadingEl = document.getElementById('loading');
        function setLoading(t) { loadingEl.className = ''; loadingEl.style.display = 'block'; loadingEl.textContent = t; }
        function fail(t) { loadingEl.className = 'error'; loadingEl.style.display = 'block'; loadingEl.textContent = t; }

        /* ------------------------------------------------------------------ *
         *  PLY parsing (typed, header-driven, little-endian binary)
         * ------------------------------------------------------------------ */
        //#PARSER_START
        const PLY_TYPES = {
            char: [1, 'getInt8'], int8: [1, 'getInt8'], uchar: [1, 'getUint8'], uint8: [1, 'getUint8'],
            short: [2, 'getInt16'], int16: [2, 'getInt16'], ushort: [2, 'getUint16'], uint16: [2, 'getUint16'],
            int: [4, 'getInt32'], int32: [4, 'getInt32'], uint: [4, 'getUint32'], uint32: [4, 'getUint32'],
            float: [4, 'getFloat32'], float32: [4, 'getFloat32'], double: [8, 'getFloat64'], float64: [8, 'getFloat64']
        };

        function parsePLYHeader(bytes) {
            const head = new TextDecoder('latin1').decode(bytes.subarray(0, Math.min(bytes.length, 1 << 16)));
            const m = /end_header\r?\n/.exec(head);
            if (!m) throw new Error('Not a valid PLY file (no end_header).');
            const lines = head.slice(0, m.index).split(/\r?\n/);
            if (lines[0].trim() !== 'ply') throw new Error('Not a PLY file.');

            let format = '', count = 0, stride = 0, elemIdx = 0, inVertex = false, vertexFirst = false;
            const props = [];
            for (const line of lines) {
                const p = line.trim().split(/\s+/);
                if (p[0] === 'format') format = p[1];
                else if (p[0] === 'element') {
                    elemIdx++;
                    if (p[1] === 'vertex') { count = parseInt(p[2], 10); inVertex = true; vertexFirst = (elemIdx === 1); }
                    else inVertex = false;
                } else if (p[0] === 'property' && inVertex) {
                    if (p[1] === 'list') throw new Error('PLY list properties are not supported on vertices.');
                    const t = PLY_TYPES[p[1]];
                    if (!t) throw new Error('Unsupported PLY property type: ' + p[1]);
                    props.push({ name: p[2], off: stride, fn: t[1], size: t[0] });
                    stride += t[0];
                }
            }
            if (format !== 'binary_little_endian') throw new Error('Only binary_little_endian PLY is supported (got ' + format + ').');
            if (!(count > 0)) throw new Error('PLY contains no vertices.');
            if (!vertexFirst) throw new Error('PLY vertex element must come first.');
            return { count, stride, props, dataStart: m.index + m[0].length };
        }

        function processPLY(bytes) {
            const hdr = parsePLYHeader(bytes);
            const N = hdr.count;
            if (hdr.dataStart + N * hdr.stride > bytes.length) throw new Error('PLY file is truncated.');
            const dv = new DataView(bytes.buffer, bytes.byteOffset, bytes.length);

            function col(name) {
                const p = hdr.props.find(q => q.name === name);
                if (!p) return null;
                const out = new Float32Array(N);
                const get = dv[p.fn].bind(dv);
                for (let i = 0, o = hdr.dataStart + p.off; i < N; i++, o += hdr.stride) out[i] = get(o, true);
                return out;
            }

            const X = col('x'), Y = col('y'), Z = col('z');
            if (!X || !Y || !Z) throw new Error('PLY has no x/y/z properties.');

            const OP = col('opacity');
            let opLogit = false;
            if (OP) {
                let mn = Infinity, mx = -Infinity;
                for (let i = 0; i < N; i++) { const v = OP[i]; if (v < mn) mn = v; if (v > mx) mx = v; }
                opLogit = (mn < 0 || mx > 1);
            }

            const S = [col('scale_0'), col('scale_1'), col('scale_2')];
            const hasScale = S.every(Boolean);
            let logScale = false;
            if (hasScale) {
                let neg = 0, cnt = 0;
                const step = Math.max(1, (N / 4000) | 0);
                for (let i = 0; i < N; i += step) { cnt++; if (S[0][i] < 0) neg++; }
                logScale = neg > cnt * 0.5;
            }

            const R = [col('rot_0'), col('rot_1'), col('rot_2'), col('rot_3')];
            const hasRot = R.every(Boolean);

            const DC = [col('f_dc_0'), col('f_dc_1'), col('f_dc_2')];
            const hasDC = DC.every(Boolean);
            let RGB = [col('red'), col('green'), col('blue')];
            if (!RGB.every(Boolean)) RGB = [col('r'), col('g'), col('b')];
            const hasRGB = RGB.every(Boolean);
            let rgbDiv = 1;
            if (!hasDC && hasRGB) {
                let mx = 0;
                for (let i = 0; i < N; i += Math.max(1, (N / 4000) | 0)) mx = Math.max(mx, RGB[0][i], RGB[1][i], RGB[2][i]);
                rgbDiv = mx > 1.5 ? 255 : 1;
            }

            const keep = new Uint32Array(N);
            const alpha = new Float32Array(N);
            let M = 0;
            for (let i = 0; i < N; i++) {
                if (!isFinite(X[i]) || !isFinite(Y[i]) || !isFinite(Z[i])) continue;
                let a = 1;
                if (OP) a = opLogit ? 1 / (1 + Math.exp(-OP[i])) : OP[i];
                if (!(a >= 1 / 255)) continue;
                alpha[i] = a > 1 ? 1 : a;
                keep[M++] = i;
            }
            if (M === 0) throw new Error('No visible splats in this model.');

            const stepS = Math.max(1, (M / 20000) | 0);
            const sx = [], sy = [], sz = [];
            for (let k = 0; k < M; k += stepS) { const i = keep[k]; sx.push(X[i]); sy.push(Y[i]); sz.push(Z[i]); }
            const med = arr => { const c = arr.slice().sort((p, q) => p - q); return c[c.length >> 1]; };
            const cx = med(sx), cy = med(sy), cz = med(sz);
            const dist = sx.map((v, j) => Math.hypot(v - cx, sy[j] - cy, sz[j] - cz)).sort((p, q) => p - q);
            const radius = dist[Math.min(dist.length - 1, (dist.length * 0.95) | 0)] || 1;
            const defScale = radius * 0.01;
            const maxScale = radius * 2;

            const W = 2048, H = Math.max(1, Math.ceil(M / W)), T = W * H * 4;
            const A = new Float32Array(T), B = new Float32Array(T), C = new Float32Array(T), D = new Float32Array(T);

            for (let k = 0; k < M; k++) {
                const i = keep[k], o = k * 4;
                A[o] = X[i] - cx; A[o + 1] = Y[i] - cy; A[o + 2] = Z[i] - cz; A[o + 3] = alpha[i];

                let s0 = defScale, s1 = defScale, s2 = defScale;
                if (hasScale) {
                    s0 = logScale ? Math.exp(S[0][i]) : S[0][i];
                    s1 = logScale ? Math.exp(S[1][i]) : S[1][i];
                    s2 = logScale ? Math.exp(S[2][i]) : S[2][i];
                    s0 = Math.min(Math.max(s0, 1e-7), maxScale);
                    s1 = Math.min(Math.max(s1, 1e-7), maxScale);
                    s2 = Math.min(Math.max(s2, 1e-7), maxScale);
                }
                let qw = 1, qx = 0, qy = 0, qz = 0;
                if (hasRot) {
                    qw = R[0][i]; qx = R[1][i]; qy = R[2][i]; qz = R[3][i];
                    const l = Math.hypot(qw, qx, qy, qz) || 1;
                    qw /= l; qx /= l; qy /= l; qz /= l;
                }
                const r00 = 1 - 2 * (qy * qy + qz * qz), r01 = 2 * (qx * qy - qw * qz), r02 = 2 * (qx * qz + qw * qy);
                const r10 = 2 * (qx * qy + qw * qz), r11 = 1 - 2 * (qx * qx + qz * qz), r12 = 2 * (qy * qz - qw * qx);
                const r20 = 2 * (qx * qz - qw * qy), r21 = 2 * (qy * qz + qw * qx), r22 = 1 - 2 * (qx * qx + qy * qy);
                const a0 = s0 * s0, b0 = s1 * s1, c0 = s2 * s2;
                B[o]     = r00 * r00 * a0 + r01 * r01 * b0 + r02 * r02 * c0;
                B[o + 1] = r00 * r10 * a0 + r01 * r11 * b0 + r02 * r12 * c0;
                B[o + 2] = r00 * r20 * a0 + r01 * r21 * b0 + r02 * r22 * c0;
                B[o + 3] = r10 * r10 * a0 + r11 * r11 * b0 + r12 * r12 * c0;
                C[o]     = r10 * r20 * a0 + r11 * r21 * b0 + r12 * r22 * c0;
                C[o + 1] = r20 * r20 * a0 + r21 * r21 * b0 + r22 * r22 * c0;

                let cr = 0.8, cg = 0.8, cb = 0.8;
                if (hasDC) {
                    cr = 0.5 + 0.28209479177387814 * DC[0][i];
                    cg = 0.5 + 0.28209479177387814 * DC[1][i];
                    cb = 0.5 + 0.28209479177387814 * DC[2][i];
                } else if (hasRGB) {
                    cr = RGB[0][i] / rgbDiv; cg = RGB[1][i] / rgbDiv; cb = RGB[2][i] / rgbDiv;
                }
                D[o] = cr < 0 ? 0 : cr > 1 ? 1 : cr;
                D[o + 1] = cg < 0 ? 0 : cg > 1 ? 1 : cg;
                D[o + 2] = cb < 0 ? 0 : cb > 1 ? 1 : cb;
                D[o + 3] = 1;
            }
            return { M, W, H, A, B, C, D, radius, total: N, logScale, opLogit };
        }

        function sortSplats(A, M, e, out, scratch) {
            const a = e[2], b = e[6], c = e[10];
            const depth = scratch.depth, keys = scratch.keys, counts = scratch.counts;
            let mn = Infinity, mx = -Infinity;
            for (let i = 0, o = 0; i < M; i++, o += 4) {
                const d = a * A[o] + b * A[o + 1] + c * A[o + 2];
                depth[i] = d;
                if (d < mn) mn = d;
                if (d > mx) mx = d;
            }
            const sc = 65535 / ((mx - mn) || 1);
            counts.fill(0);
            for (let i = 0; i < M; i++) { const k = ((depth[i] - mn) * sc) | 0; keys[i] = k; counts[k]++; }
            let acc = 0;
            for (let k = 0; k < 65536; k++) { const n = counts[k]; counts[k] = acc; acc += n; }
            for (let i = 0; i < M; i++) out[counts[keys[i]]++] = i;
        }
        //#PARSER_END

        async function fetchBytes(url, onProgress) {
            const resp = await fetch(url);
            if (!resp.ok) throw new Error('Download failed (HTTP ' + resp.status + ')');
            if (!resp.body || !resp.body.getReader) return new Uint8Array(await resp.arrayBuffer());
            const total = parseInt(resp.headers.get('Content-Length') || '0', 10);
            let buf = new Uint8Array(total > 0 ? total : 1 << 24);
            let got = 0;
            const reader = resp.body.getReader();
            for (;;) {
                const { done, value } = await reader.read();
                if (done) break;
                if (got + value.length > buf.length) {
                    const nb = new Uint8Array(Math.max(buf.length * 2, got + value.length));
                    nb.set(buf.subarray(0, got));
                    buf = nb;
                }
                buf.set(value, got);
                got += value.length;
                onProgress(got, total);
            }
            return buf.subarray(0, got);
        }
        const nextFrame = () => new Promise(r => requestAnimationFrame(() => setTimeout(r, 0)));

        if (typeof THREE === 'undefined' || !THREE.OrbitControls) {
            fail('Could not load three.js from the CDN. Check your network / ad-blocker and reload.');
            return;
        }
        const modelUrl = new URLSearchParams(window.location.search).get('model');
        const canvas = document.getElementById('canvas');

        let renderer;
        try {
            renderer = new THREE.WebGLRenderer({ canvas, antialias: false, alpha: false, powerPreference: 'high-performance' });
        } catch (e) { fail('WebGL is not available in this browser.'); return; }
        if (!renderer.capabilities.isWebGL2) { fail('WebGL2 is required for the Gaussian Splat renderer.'); return; }
        renderer.setPixelRatio(Math.min(window.devicePixelRatio || 1, 2));
        canvas.addEventListener('webglcontextlost', e => { e.preventDefault(); fail('WebGL context lost. Reload the viewer.'); });

        const scene = new THREE.Scene();
        scene.background = new THREE.Color(0x04060a);
        const camera = new THREE.PerspectiveCamera(50, 1, 0.01, 1000);
        const controls = new THREE.OrbitControls(camera, canvas);
        controls.enableDamping = true;
        controls.dampingFactor = 0.07;
        controls.screenSpacePanning = true;
        controls.autoRotateSpeed = 1.5;

        const grid = new THREE.GridHelper(4, 20, 0x1e293b, 0x0c1322);
        scene.add(grid);

        function resize() {
            const w = Math.max(1, window.innerWidth), h = Math.max(1, window.innerHeight);
            renderer.setSize(w, h, false);
            camera.aspect = w / h;
            camera.updateProjectionMatrix();
        }
        window.addEventListener('resize', resize);
        if (window.ResizeObserver) new ResizeObserver(resize).observe(document.body);
        resize();

        const VERT = `
            precision highp float;
            precision highp sampler2D;
            uniform sampler2D tA;
            uniform sampler2D tB;
            uniform sampler2D tC;
            uniform sampler2D tD;
            uniform vec2 texSize;
            uniform vec2 viewport;
            uniform float uNear;
            uniform float splatScale;
            attribute float splatIndex;
            varying vec4 vColor;
            varying vec2 vCorner;

            void main() {
                float row = floor(splatIndex / texSize.x);
                float colx = splatIndex - row * texSize.x;
                vec2 uv = vec2((colx + 0.5) / texSize.x, (row + 0.5) / texSize.y);

                vec4 pa = texture2D(tA, uv);
                vec4 viewPos = modelViewMatrix * vec4(pa.xyz, 1.0);
                float d = -viewPos.z;
                vec4 clip = projectionMatrix * viewPos;

                if (d < uNear || abs(clip.x) > 1.3 * clip.w || abs(clip.y) > 1.3 * clip.w) {
                    gl_Position = vec4(0.0, 0.0, 2.0, 1.0);
                    vColor = vec4(0.0);
                    vCorner = vec2(0.0);
                    return;
                }

                vec4 c0 = texture2D(tB, uv);
                vec4 c1 = texture2D(tC, uv);
                mat3 V = mat3(c0.x, c0.y, c0.z,
                              c0.y, c0.w, c1.x,
                              c0.z, c1.x, c1.y);
                mat3 W = mat3(modelViewMatrix);

                float fx = projectionMatrix[0][0] * viewport.x * 0.5;
                float fy = projectionMatrix[1][1] * viewport.y * 0.5;
                float limx = 1.3 / projectionMatrix[0][0];
                float limy = 1.3 / projectionMatrix[1][1];
                float tx = clamp(viewPos.x / d, -limx, limx) * d;
                float ty = clamp(viewPos.y / d, -limy, limy) * d;

                vec3 j0 = vec3(fx / d, 0.0, fx * tx / (d * d));
                vec3 j1 = vec3(0.0, fy / d, fy * ty / (d * d));
                vec3 T0 = j0 * W;
                vec3 T1 = j1 * W;

                float a = dot(T0, V * T0) + 0.3;
                float b = dot(T0, V * T1);
                float c = dot(T1, V * T1) + 0.3;

                float mid = 0.5 * (a + c);
                float rad = length(vec2(0.5 * (a - c), b));
                float l1 = mid + rad;
                float l2 = max(mid - rad, 0.1);
                vec2 ev = vec2(b, l1 - a);
                float el = length(ev);
                ev = el > 1e-6 ? ev / el : vec2(1.0, 0.0);

                vec2 major = min(sqrt(2.0 * l1), 1024.0) * ev;
                vec2 minor = min(sqrt(2.0 * l2), 1024.0) * vec2(ev.y, -ev.x);
                vec2 offs = (position.x * major + position.y * minor) * splatScale;

                vec2 ndc = clip.xy / clip.w + offs * 2.0 / viewport;
                gl_Position = vec4(ndc, clip.z / clip.w, 1.0);
                vColor = vec4(texture2D(tD, uv).rgb, pa.w);
                vCorner = position.xy;
            }`;

        const FRAG = `
            precision highp float;
            varying vec4 vColor;
            varying vec2 vCorner;
            void main() {
                float A = -dot(vCorner, vCorner);
                if (A < -4.0) discard;
                float B = exp(A) * vColor.a;
                if (B < 0.004) discard;
                gl_FragColor = vec4(vColor.rgb * B, B);
            }`;

        let mesh = null, model = null, flipped = true, forceSort = true;
        let attr = null, scratch = null, uniforms = null;
        const mv = new THREE.Matrix4();
        const lastDir = [0, 0, 0];
        const home = { pos: new THREE.Vector3(), dist: 1 };
        const vp = new THREE.Vector2();
        let lastSortTime = 0;

        function makeTexture(data, W, H) {
            const t = new THREE.DataTexture(data, W, H, THREE.RGBAFormat, THREE.FloatType);
            t.minFilter = THREE.NearestFilter;
            t.magFilter = THREE.NearestFilter;
            t.generateMipmaps = false;
            t.needsUpdate = true;
            return t;
        }

        function buildScene(m) {
            model = m;
            const geo = new THREE.InstancedBufferGeometry();
            geo.setAttribute('position', new THREE.BufferAttribute(new Float32Array([-2, -2, 0, 2, -2, 0, 2, 2, 0, -2, 2, 0]), 3));
            geo.setIndex([0, 1, 2, 0, 2, 3]);
            const idx = new Float32Array(m.M);
            for (let i = 0; i < m.M; i++) idx[i] = i;
            attr = new THREE.InstancedBufferAttribute(idx, 1);
            attr.setUsage(THREE.DynamicDrawUsage);
            geo.setAttribute('splatIndex', attr);
            geo.instanceCount = m.M;

            uniforms = {
                tA: { value: makeTexture(m.A, m.W, m.H) },
                tB: { value: makeTexture(m.B, m.W, m.H) },
                tC: { value: makeTexture(m.C, m.W, m.H) },
                tD: { value: makeTexture(m.D, m.W, m.H) },
                texSize: { value: new THREE.Vector2(m.W, m.H) },
                viewport: { value: new THREE.Vector2(1, 1) },
                uNear: { value: camera.near },
                splatScale: { value: 1.0 }
            };
            const mat = new THREE.ShaderMaterial({
                uniforms, vertexShader: VERT, fragmentShader: FRAG,
                transparent: true, depthTest: false, depthWrite: false, side: THREE.DoubleSide,
                blending: THREE.CustomBlending, blendEquation: THREE.AddEquation,
                blendSrc: THREE.OneFactor, blendDst: THREE.OneMinusSrcAlphaFactor
            });
            mesh = new THREE.Mesh(geo, mat);
            mesh.frustumCulled = false;
            mesh.rotation.x = flipped ? Math.PI : 0;
            scene.add(mesh);

            scratch = { depth: new Float32Array(m.M), keys: new Uint16Array(m.M), counts: new Uint32Array(65536) };

            const vfov = THREE.MathUtils.degToRad(camera.fov);
            const hfov = 2 * Math.atan(Math.tan(vfov / 2) * camera.aspect);
            const dist = (m.radius / Math.sin(Math.min(vfov, hfov) / 2)) * 0.9;
            home.dist = dist;
            home.pos.set(0, dist * 0.18, dist);
            camera.near = Math.max(dist * 0.01, 1e-4);
            camera.far = dist * 60;
            camera.updateProjectionMatrix();
            controls.minDistance = dist * 0.05;
            controls.maxDistance = dist * 12;
            resetCamera();
            grid.scale.setScalar(Math.max(m.radius, 1e-3) / 2);
            grid.position.y = -m.radius * 1.05;

            document.getElementById('count').textContent = m.M.toLocaleString() + ' splats';
            document.getElementById('bar').style.display = 'flex';
        }

        function resetCamera() {
            camera.position.copy(home.pos);
            controls.target.set(0, 0, 0);
            controls.update();
        }

        function maybeSort(now) {
            mesh.updateMatrixWorld();
            mv.copy(camera.matrixWorld).invert().multiply(mesh.matrixWorld);
            const e = mv.elements;
            const changed = forceSort ||
                Math.abs(e[2] - lastDir[0]) > 5e-4 || Math.abs(e[6] - lastDir[1]) > 5e-4 || Math.abs(e[10] - lastDir[2]) > 5e-4;
            if (!changed) return;
            if (!forceSort && now - lastSortTime < 30) return;
            sortSplats(model.A, model.M, e, attr.array, scratch);
            attr.needsUpdate = true;
            lastDir[0] = e[2]; lastDir[1] = e[6]; lastDir[2] = e[10];
            forceSort = false;
            lastSortTime = performance.now();
        }

        function frame(now) {
            requestAnimationFrame(frame);
            controls.update();
            camera.updateMatrixWorld();
            if (mesh) {
                maybeSort(now);
                renderer.getDrawingBufferSize(vp);
                uniforms.viewport.value.copy(vp);
                uniforms.uNear.value = camera.near;
            }
            renderer.render(scene, camera);
        }
        requestAnimationFrame(frame);

        document.getElementById('btnReset').onclick = resetCamera;
        document.getElementById('btnRotate').onclick = function () {
            controls.autoRotate = !controls.autoRotate;
            this.classList.toggle('on', controls.autoRotate);
        };
        document.getElementById('btnFlip').onclick = function () {
            if (!mesh) return;
            flipped = !flipped;
            mesh.rotation.x = flipped ? Math.PI : 0;
            this.classList.toggle('on', !flipped);
            forceSort = true;
        };
        document.getElementById('sizeSlider').oninput = function () { if (uniforms) uniforms.splatScale.value = parseFloat(this.value); };

        (async function main() {
            if (!modelUrl) { fail('No model specified (?model=...).'); return; }
            try {
                setLoading('Downloading model...');
                const bytes = await fetchBytes(modelUrl, (got, total) => {
                    const mb = (got / 1048576).toFixed(1);
                    setLoading(total ? 'Downloading model... ' + Math.round(got / total * 100) + '% (' + mb + ' MB)' : 'Downloading model... ' + mb + ' MB');
                });
                setLoading('Preparing splats...');
                await nextFrame();
                const m = processPLY(bytes);
                await nextFrame();
                buildScene(m);
                loadingEl.style.display = 'none';
            } catch (err) {
                console.error(err);
                fail('Failed to load model: ' + err.message);
            }
        })();
    })();
    </script>
</body>
</html>
"""

def html_response(body):
    resp = Response(body, mimetype="text/html")
    resp.headers["Cache-Control"] = "no-store"
    return resp

@app.route("/")
def index():
    return html_response(HTML_STUDIO)

@app.route("/viewer")
def viewer():
    return html_response(HTML_VIEWER)

ALLOWED_GAUSSIANS = {65536, 131072, 262144, 524288, 1048576}

@app.route("/generate", methods=["POST"])
def generate_handler():
    if "image" not in request.files:
        return jsonify({"success": False, "error": "No image provided"}), 200

    try:
        mode = request.form.get("mode", "real")
        if mode not in ("real", "synthetic"):
            mode = "real"
        num_gaussians = int(request.form.get("num_gaussians", 262144))
        if num_gaussians not in ALLOWED_GAUSSIANS:
            num_gaussians = 262144
        steps = max(1, min(150, int(request.form.get("steps", 35))))
        guidance_scale = max(0.0, min(20.0, float(request.form.get("guidance_scale", 3.0))))
        time_shift = max(0.1, min(10.0, float(request.form.get("time_shift", 3.0))))
        seed_raw = (request.form.get("seed") or "42").strip()
        seed = int(float(seed_raw))
    except (TypeError, ValueError):
        return jsonify({"success": False, "error": "Invalid numeric parameter"}), 200
    if seed < 0:
        seed = int(time.time() * 1000) % 2_147_483_647

    task_id = uuid.uuid4().hex[:8]
    input_path = os.path.join(UPLOAD_DIR, f"in_{task_id}.png")

    try:
        img = Image.open(request.files["image"].stream)
        img = ImageOps.exif_transpose(img)
        if img.mode in ("RGBA", "LA", "PA") or (img.mode == "P" and "transparency" in img.info):
            rgba = img.convert("RGBA")
            bg = Image.new("RGBA", rgba.size, (255, 255, 255, 255))
            bg.alpha_composite(rgba)
            img = bg
        img = img.convert("RGB")
        if max(img.size) > 4096:
            img.thumbnail((4096, 4096), Image.LANCZOS)
        img.save(input_path)
    except Exception as e:
        return jsonify({"success": False, "error": f"Invalid image file: {e}"}), 200

    set_task(task_id, status="queued")
    threading.Thread(
        target=run_gpu_generation,
        args=(task_id, input_path, mode, num_gaussians, steps, guidance_scale, time_shift, seed),
        daemon=True
    ).start()

    return jsonify({"success": True, "task_id": task_id}), 200

@app.route("/status/<task_id>")
def status_handler(task_id):
    with TASK_LOCK:
        task = TASKS.get(task_id)
    resp = jsonify(task or {"status": "failed", "error": "Task not found (server was restarted?)"})
    resp.headers["Cache-Control"] = "no-store"
    return resp, 200

@app.route("/files/<path:filename>")
def serve_file(filename):
    return send_from_directory(OUTPUT_DIR, filename, conditional=True)

# 6. Start Flask in a background thread
_idol_server = make_server("127.0.0.1", PORT, app, threaded=True)
threading.Thread(target=_idol_server.serve_forever, daemon=True).start()

# 7. Launch Cloudflare Tunnel
TUNNEL_LOG = "/content/cloudflared.log"
_tunnel_log = open(TUNNEL_LOG, "w")
_idol_tunnel = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--protocol", "http2", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=_tunnel_log, stderr=subprocess.STDOUT
)

public_url = None
t_tunnel = time.time()
while time.time() - t_tunnel < 60 and _idol_tunnel.poll() is None:
    try:
        with open(TUNNEL_LOG, "r") as f:
            m = re.search(r"https://(?!api\.)[a-zA-Z0-9-]+\.trycloudflare\.com", f.read())
    except OSError:
        m = None
    if m:
        public_url = m.group(0)
        break
    time.sleep(0.5)

W = 74
print(f"\n\033[1;32m┌{'─' * W}┐\033[0m")
print(f"\033[1;32m│\033[0m  \033[1;37m{'🚀 IDOL 3D IS LIVE':<{W - 4}}\033[0m\033[1;32m│\033[0m")
print(f"\033[1;32m└{'─' * W}┘\033[0m")
if public_url:
    print(f"  \033[1;36mPublic URL:\033[0m \033[4;1;37m{public_url}\033[0m")
    print("  \033[0;90mThe link can take ~10-30s to start resolving. Anyone with it can use your GPU - don't share it publicly.\033[0m")
else:
    print(f"  \033[1;31mCould not obtain a Cloudflare URL.\033[0m See {TUNNEL_LOG} (tail below), then re-run this cell:")
    try:
        print("".join(open(TUNNEL_LOG).readlines()[-8:]))
    except OSError:
        pass
print("  \033[0;90mKeep this cell running. Interrupt it (■) to shut the server and tunnel down.\033[0m\n")

# 8. Keep the cell alive; clean up on interrupt
try:
    while True:
        time.sleep(1)
        if _idol_tunnel.poll() is not None:
            print("\033[1;31m[TUNNEL]\033[0m cloudflared exited. Re-run this cell to get a new URL.")
            break
except KeyboardInterrupt:
    print("\n\033[1;33m[STOP]\033[0m Shutting down...")
finally:
    _idol_server.shutdown()
    if _idol_tunnel.poll() is None:
        _idol_tunnel.terminate()
    _tunnel_log.close()
